In [1]:
from IPython.display import display, HTML, clear_output

clear_output(wait=True)

SYMBOLS = {
    'K': '♔', 'Q': '♕', 'R': '♖', 'B': '♗', 'N': '♘', 'P': '♙',
    'k': '♚', 'q': '♛', 'r': '♜', 'b': '♝', 'n': '♞', 'p': '♟',
    ' ': ''
}

def create_board():
    return [
        ['r','n','b','q','k','b','n','r'],
        ['p','p','p','p','p','p','p','p'],
        [' ',' ',' ',' ',' ',' ',' ',' '],
        [' ',' ',' ',' ',' ',' ',' ',' '],
        [' ',' ',' ',' ',' ',' ',' ',' '],
        [' ',' ',' ',' ',' ',' ',' ',' '],
        ['P','P','P','P','P','P','P','P'],
        ['R','N','B','Q','K','B','N','R'],
    ]

def is_white(piece):
    return piece != ' ' and piece.isupper()

def is_black(piece):
    return piece != ' ' and piece.islower()

def is_empty(piece):
    return piece == ' '

def is_on_board(row, col):
    return 0 <= row < 8 and 0 <= col < 8

def is_enemy(piece, white_turn):
    return is_black(piece) if white_turn else is_white(piece)

def is_friend(piece, white_turn):
    return is_white(piece) if white_turn else is_black(piece)

def square_to_position(square):
    files = "abcdefgh"
    col = files.index(square[0])
    row = 8 - int(square[1])
    return row, col

def position_to_square(row, col):
    files = "abcdefgh"
    return files[col] + str(8 - row)

def get_moves(board, row, col):
    piece = board[row][col]
    if piece == ' ':
        return []

    white = is_white(piece)
    kind = piece.lower()
    moves = []

    if kind == 'p':
        direction = -1 if white else 1
        start_row = 6 if white else 1

        r1 = row + direction
        if is_on_board(r1, col) and board[r1][col] == ' ':
            moves.append((r1, col))

            r2 = row + 2 * direction
            if row == start_row and board[r2][col] == ' ':
                moves.append((r2, col))

        for dc in [-1, 1]:
            r, c = row + direction, col + dc
            if is_on_board(r, c) and is_enemy(board[r][c], white):
                moves.append((r, c))

    elif kind == 'n':
        jumps = [
            (-2,-1), (-2,1), (-1,-2), (-1,2),
            (1,-2), (1,2), (2,-1), (2,1)
        ]
        for dr, dc in jumps:
            r, c = row + dr, col + dc
            if is_on_board(r, c) and not is_friend(board[r][c], white):
                moves.append((r, c))

    elif kind == 'r':
        directions = [(0,1), (0,-1), (1,0), (-1,0)]
        for dr, dc in directions:
            r, c = row + dr, col + dc
            while is_on_board(r, c):
                if is_friend(board[r][c], white):
                    break
                moves.append((r, c))
                if is_enemy(board[r][c], white):
                    break
                r += dr
                c += dc

    elif kind == 'b':
        directions = [(1,1), (1,-1), (-1,1), (-1,-1)]
        for dr, dc in directions:
            r, c = row + dr, col + dc
            while is_on_board(r, c):
                if is_friend(board[r][c], white):
                    break
                moves.append((r, c))
                if is_enemy(board[r][c], white):
                    break
                r += dr
                c += dc

    elif kind == 'q':
        directions = [
            (0,1), (0,-1), (1,0), (-1,0),
            (1,1), (1,-1), (-1,1), (-1,-1)
        ]
        for dr, dc in directions:
            r, c = row + dr, col + dc
            while is_on_board(r, c):
                if is_friend(board[r][c], white):
                    break
                moves.append((r, c))
                if is_enemy(board[r][c], white):
                    break
                r += dr
                c += dc

    elif kind == 'k':
        directions = [
            (-1,-1), (-1,0), (-1,1),
            (0,-1),          (0,1),
            (1,-1),  (1,0),  (1,1)
        ]
        for dr, dc in directions:
            r, c = row + dr, col + dc
            if is_on_board(r, c) and not is_friend(board[r][c], white):
                moves.append((r, c))

    return moves

class ChessGame:
    def __init__(self):
        self.board = create_board()
        self.white_turn = True
        self.history = []
        self.last_from = None
        self.last_to = None

    def show_board(self, message=""):
        files = ['a','b','c','d','e','f','g','h']

        html = """
        <h2>♟ Chess Adventure</h2>
        <p><b>Turn:</b> """ + ("White" if self.white_turn else "Black") + """</p>
        """

        if message:
            html += f"<p style='color:darkblue; font-size:16px;'><b>{message}</b></p>"

        html += """
        <style>
        .chess-board {
            border-collapse: collapse;
            margin-top: 10px;
            font-size: 34px;
        }
        .chess-board td {
            width: 62px;
            height: 62px;
            text-align: center;
            vertical-align: middle;
            border: 1px solid #555;
        }
        .label {
            width: 25px;
            height: 25px;
            background: white;
            color: black;
            font-size: 14px;
            font-weight: bold;
        }
        </style>
        <table class="chess-board">
        """

        html += "<tr><td class='label'></td>"
        for f in files:
            html += f"<td class='label'>{f}</td>"
        html += "</tr>"

        for r in range(8):
            html += f"<tr><td class='label'>{8-r}</td>"
            for c in range(8):
                piece = self.board[r][c]
                color = "#F0D9B5" if (r+c) % 2 == 0 else "#B58863"

                if self.last_from == (r, c):
                    color = "#F6F669"
                elif self.last_to == (r, c):
                    color = "#7FC97F"

                html += f"<td style='background:{color};'>{SYMBOLS[piece]}</td>"
            html += f"<td class='label'>{8-r}</td></tr>"

        html += "<tr><td class='label'></td>"
        for f in files:
            html += f"<td class='label'>{f}</td>"
        html += "</tr></table>"

        html += """
        <h3>How to play in Kaggle</h3>
        <p>Run moves in a new cell like this:</p>
        <pre>game.move("e2", "e4")</pre>
        <pre>game.move("e7", "e5")</pre>
        <pre>game.show_moves("g1")</pre>
        """

        if self.history:
            html += "<h3>Move History</h3><ol>"
            for move in self.history:
                html += f"<li>{move}</li>"
            html += "</ol>"

        display(HTML(html))

    def show_moves(self, square):
        row, col = square_to_position(square)
        piece = self.board[row][col]

        if piece == ' ':
            self.show_board(f"{square} is empty.")
            return

        white_piece = is_white(piece)
        if self.white_turn != white_piece:
            self.show_board("That is not this player's turn.")
            return

        moves = get_moves(self.board, row, col)
        move_names = [position_to_square(r, c) for r, c in moves]

        if move_names:
            self.show_board(f"{SYMBOLS[piece]} at {square} can move to: {', '.join(move_names)}")
        else:
            self.show_board(f"{SYMBOLS[piece]} at {square} has no legal moves.")

    def move(self, from_square, to_square):
        fr, fc = square_to_position(from_square)
        tr, tc = square_to_position(to_square)

        piece = self.board[fr][fc]

        if piece == ' ':
            self.show_board(f"No piece on {from_square}.")
            return

        if self.white_turn and not is_white(piece):
            self.show_board("It is White's turn.")
            return

        if not self.white_turn and not is_black(piece):
            self.show_board("It is Black's turn.")
            return

        legal_moves = get_moves(self.board, fr, fc)

        if (tr, tc) not in legal_moves:
            self.show_board(f"Illegal move: {from_square} to {to_square}.")
            return

        captured = self.board[tr][tc]

        self.board[tr][tc] = piece
        self.board[fr][fc] = ' '

        if piece == 'P' and tr == 0:
            self.board[tr][tc] = 'Q'
            piece = 'Q'

        if piece == 'p' and tr == 7:
            self.board[tr][tc] = 'q'
            piece = 'q'

        self.last_from = (fr, fc)
        self.last_to = (tr, tc)

        capture_text = ""
        if captured != ' ':
            capture_text = f" captured {SYMBOLS[captured]}"

        self.history.append(
            f"{'White' if self.white_turn else 'Black'}: {SYMBOLS[piece]} {from_square} → {to_square}{capture_text}"
        )

        self.white_turn = not self.white_turn
        self.show_board(f"Moved {SYMBOLS[piece]} from {from_square} to {to_square}.")

    def reset(self):
        self.board = create_board()
        self.white_turn = True
        self.history = []
        self.last_from = None
        self.last_to = None
        self.show_board("New game started. White goes first.")

print("Welcome to Chess Adventure!")
print("=" * 40)
print("This Kaggle version has NO VBox sentence.")
print("To move, run: game.move('e2', 'e4')")
print("To see moves, run: game.show_moves('g1')")
print("To restart, run: game.reset()")
print("=" * 40)

game = ChessGame()
game.show_board("White goes first.")

Welcome to Chess Adventure!
This Kaggle version has NO VBox sentence.
To move, run: game.move('e2', 'e4')
To see moves, run: game.show_moves('g1')
To restart, run: game.reset()


In [2]:
game.move("e2", "e4")

In [3]:
game.move("e7", "e5")